# Image eval demo

This notebook grades image model outputs the way the [OpenAI image evals cookbook](https://developers.openai.com/cookbook/examples/multimodal/image_evals) does, with [Pixeltable](https://www.pixeltable.com/) as the harness.

The harness is two tables and two views:

- `generations`: text-to-image rows. The `image` column is produced by the built-in `openai.image_generations`.
- `edits`: image-edit rows. The `image` column is produced by the built-in `gemini.generate_content`.
- `gen_evals` and `edit_evals`: views over the `eval_ready` rows of each table. They run the vision judge and return a verdict, scores, and tags.

Everything here is a Pixeltable query. When you select the `image` column, the result renders the picture inline next to its scores.

Run `scripts/seed_demo.py` first. That inserts the four demo cases, which generates their images on insert, so it calls OpenAI and Gemini. You need `OPENAI_API_KEY` and `GEMINI_API_KEY` in `.env`.

In [ ]:
import udfs  # registers the UDFs; import this before you touch the tables
import pixeltable as pxt

generations = pxt.get_table('img_eval/generations')
edits = pxt.get_table('img_eval/edits')
gen_evals = pxt.get_table('img_eval/gen_evals')
edit_evals = pxt.get_table('img_eval/edit_evals')

generations.where(generations.batch_id == 'demo_v1').select(
    generations.image, generations.case_id, generations.workflow
).collect()

The seeded batch already has images, because inserting a row runs the image model. `eval_ready` is still `False`, so nothing is graded yet. Here are the two edits.

In [ ]:
edits.where(edits.batch_id == 'demo_v1').select(
    edits.image, edits.case_id, edits.workflow
).collect()

## Grade the batch

Set `eval_ready` to `True` on both tables. That moves the rows into the eval views and runs the judge. This calls OpenAI and can take a minute.

In [ ]:
generations.update({'eval_ready': True}, where=generations.batch_id == 'demo_v1')
edits.update({'eval_ready': True}, where=edits.batch_id == 'demo_v1')

gen_evals.where(gen_evals.batch_id == 'demo_v1').select(
    gen_evals.image, gen_evals.case_id, gen_evals.verdict,
    gen_evals.scores, gen_evals.reason, gen_evals.tags,
).collect()

The same view for the edits.

In [ ]:
edit_evals.where(edit_evals.batch_id == 'demo_v1').select(
    edit_evals.image, edit_evals.case_id, edit_evals.verdict,
    edit_evals.scores, edit_evals.reason, edit_evals.tags,
).collect()

### What each job checks

- **UI mockup.** The screen must match the request, the required labels must be present and readable, and the layout and controls must be usable. Instruction following and in-image text are gates. Layout and affordances are graded from 0 to 5.
- **Marketing flyer.** On top of the judge, a vision pass reads the text in the image and `text_check` compares it against the required copy list. A missing required string fails the flyer even if the design is good. Extra text, such as the shop name, is reported but does not fail the check.
- **Virtual try-on.** The output must keep the same person, apply the reference garment, and preserve body shape. Each dimension is scored from 0 to 5.
- **Logo edit.** This job fails on near misses. A common outcome is that the text change is correct but the edit shifted the background or style a little, so `non_target_invariance` drops and the output fails on spill. That is expected with these thresholds.

Here is the marketing exact-copy check on its own. `extracted_text` is what the vision pass read from the flyer, and `text_check` reports whether every required string was found.

In [ ]:
gen_evals.where(gen_evals.case_id == 'coffee_flyer_generation').select(
    gen_evals.extracted_text, gen_evals.text_check
).collect()

## Generate and grade a new image

Insert a row into `generations`. Pixeltable runs `openai.image_generations` on insert. The model and size are set once in `schema.py`, so you do not pass them here.

Each row has two prompt fields. `gen_prompt` is sent to the image model. `prompt` is the instruction shown to the judge. They often match, as they do here.

In [ ]:
_ui_prompt = (
    'Generate a high-fidelity mobile checkout screen for an ecommerce app. '
    'Orientation: portrait. Include labels Checkout, Place Order, Edit Cart.'
)

generations.insert(
    case_id='ui_checkout_v2',
    batch_id='run_2',
    workflow='ui_mockup',
    gen_prompt=_ui_prompt,
    prompt=_ui_prompt,
    criteria='Mobile checkout; required labels present; clear primary call to action.',
    eval_ready=True,
)

gen_evals.where(gen_evals.case_id == 'ui_checkout_v2').select(
    gen_evals.image, gen_evals.verdict, gen_evals.scores, gen_evals.reason, gen_evals.tags
).collect()

## Edit and grade a new image

Insert a row into `edits`. Pixeltable runs `gemini.generate_content` on insert. Editing uses Gemini because it takes several reference images in one call, which virtual try-on needs. The README notes the plan to move editing to OpenAI once the multi-image edit path is confirmed against the live API.

In [ ]:
_logo_prompt = (
    'Edit the logo by changing the text from FIELD to BUTTER.\n'
    'Do not change any other text, colors, shapes, or layout.'
)

edits.insert(
    case_id='logo_year_edit_live',
    batch_id='run_edit_logo',
    workflow='logo_edit',
    gen_prompt=_logo_prompt,
    prompt=_logo_prompt,
    criteria=(
        'The requested edit is applied exactly.\n'
        'All non-target content remains unchanged.\n'
        'Character style, color, and geometry remain consistent with the original.'
    ),
    ref_image='use_cases/logo_edit/assets/logo_input.png',
    eval_ready=True,
)

edit_evals.where(edit_evals.case_id == 'logo_year_edit_live').select(
    edit_evals.image, edit_evals.verdict, edit_evals.scores, edit_evals.reason, edit_evals.tags
).collect()

In [ ]:
_vto_prompt = (
    'Put the person in the first image into the jacket shown in the second image.\n'
    "Keep the person's face, pose, body shape, and background unchanged.\n"
    "Preserve the garment's color, pattern, and key details.\n"
    'Do not add extra accessories, text, or new elements.'
)

edits.insert(
    case_id='vto_jacket_tryon_live',
    batch_id='run_edit_vto',
    workflow='virtual_try_on',
    gen_prompt=_vto_prompt,
    prompt=_vto_prompt,
    criteria=(
        'The output preserves the same person and background.\n'
        'The jacket matches the reference garment closely.\n'
        'Body shape and pose remain consistent outside normal garment effects.\n'
        'The result looks physically plausible.'
    ),
    ref_image='use_cases/virtual_try_on/assets/person.png',
    ref_image_2='use_cases/virtual_try_on/assets/garment.png',
    eval_ready=True,
)

edit_evals.where(edit_evals.case_id == 'vto_jacket_tryon_live').select(
    edit_evals.image, edit_evals.verdict, edit_evals.scores, edit_evals.reason, edit_evals.tags
).collect()

## Generate synthetic test data

The cells above are also how you test the harness. You do not need real images to exercise a rubric. Generate inputs and outputs with the same built-in functions, mark them ready, and check that the verdicts and tags match what you expect. A rubric that passes a clean case and fails a broken one is one you can rely on.

[`scripts/make_tryon_inputs.py`](scripts/make_tryon_inputs.py) shows the pattern. It declares a small class-based table whose image is computed by `openai.image_generations`, generates the person and the garment, and saves them as the try-on reference images. So the demo references are model-generated, not from an unknown stock source.

## Re-grade without regenerating

The judge output is cached. To re-run the judge, for example after you change a rubric, recompute the `judge_raw` column. The verdict, scores, and tags recompute from it.

In [ ]:
gen_evals.recompute_columns(columns=['judge_raw'], where=gen_evals.batch_id == 'demo_v1')

gen_evals.where(gen_evals.batch_id == 'demo_v1').select(
    gen_evals.image, gen_evals.case_id, gen_evals.verdict, gen_evals.tags
).collect()

## Reset the sample batch

Delete the demo rows from both tables, then re-run `uv run python scripts/seed_demo.py` from the terminal to generate and grade again.

In [ ]:
generations.delete(where=generations.batch_id == 'demo_v1')
edits.delete(where=edits.batch_id == 'demo_v1')